# 12 — Model Monitoring & Evaluation

This notebook implements the **Monitoring — Model Degradation Monitoring** requirement.

The project now contains:
- Failure prediction
- RUL prediction
- Time-series sensor forecasting

This notebook evaluates model health using prediction performance, forecasting error, feature drift, prediction drift, and configurable health thresholds.

> Machine monitoring asks **"Is the machine healthy?"**  
> Model monitoring asks **"Is our AI model still reliable?"**


In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error
)
from scipy.stats import ks_2samp

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

SMART_TWIN_PATH = "../data/processed/iot_smart_twin_results.csv"
REFERENCE_PATH = "../data/all_machines_combined.csv"
FORECAST_PATH = "../data/processed/time_series_forecasts.csv"
FORECAST_METRICS_PATH = "../data/processed/time_series_forecast_metrics.csv"
OUTPUT_DIR = "../data/processed"

def safe_read(path):
    return pd.read_csv(path) if os.path.exists(path) else None

smart_twin = safe_read(SMART_TWIN_PATH)
reference_df = safe_read(REFERENCE_PATH)
forecast_df = safe_read(FORECAST_PATH)
forecast_metrics = safe_read(FORECAST_METRICS_PATH)

for name, data in [
    ("Smart Twin", smart_twin),
    ("Reference", reference_df),
    ("Forecasts", forecast_df),
    ("Forecast Metrics", forecast_metrics)
]:
    print(f"{name}: {None if data is None else data.shape}")


Smart Twin: (1930, 24)
Reference: None
Forecasts: (1326, 8)
Forecast Metrics: (21, 6)


## STEP 1 — Inspect schemas

The notebook checks the actual artifacts before calculating metrics. It does not invent metrics when the required columns are unavailable.


In [2]:
for name, data in [
    ("Smart Twin", smart_twin),
    ("Reference", reference_df),
    ("Forecasts", forecast_df),
    ("Forecast Metrics", forecast_metrics)
]:
    if data is not None:
        print(f"\n{name} columns:")
        print(data.columns.tolist())



Smart Twin columns:
['Timestamp', 'Machine_ID', 'Machine_Type', 'Cycle', 'Temperature_C', 'Vibration_mm_s', 'Pressure_bar', 'Rotational_Speed_RPM', 'Load_Fraction', 'Torque_Nm', 'Current_A', 'Acoustic_dB', 'Wear_Level', 'Health_Index', 'Anomaly_Score', 'Simulation_Fault', 'failure_prediction', 'failure_probability', 'failure_risk', 'predicted_rul_cycles', 'rul_status', 'overall_status', 'maintenance_action', 'health_score']

Forecasts columns:
['Machine_ID', 'Cycle', 'Sensor', 'Actual_Value', 'Predicted_Value', 'Absolute_Error', 'Prediction_Type', 'Forecast_Horizon']

Forecast Metrics columns:
['Machine_ID', 'Sensor', 'Train_Rows', 'Test_Rows', 'MAE', 'RMSE']


## STEP 2 — Time-series model monitoring

The previous notebook generated MAE/RMSE for every machine and sensor. These metrics become the baseline for monitoring.

Thresholds are configurable project-level demonstration thresholds:
- 1.25× baseline → WARNING
- 1.50× baseline → DEGRADING


In [3]:
WARNING_MULTIPLIER = 1.25
DEGRADING_MULTIPLIER = 1.50

if forecast_metrics is not None and not forecast_metrics.empty:
    baseline_mae = forecast_metrics["MAE"].mean()
    baseline_rmse = forecast_metrics["RMSE"].mean()

    forecast_monitoring = forecast_metrics.copy()
    forecast_monitoring["MAE_Ratio_To_Baseline"] = forecast_monitoring["MAE"] / baseline_mae
    forecast_monitoring["RMSE_Ratio_To_Baseline"] = forecast_monitoring["RMSE"] / baseline_rmse

    def forecast_status(row):
        ratio = max(row["MAE_Ratio_To_Baseline"], row["RMSE_Ratio_To_Baseline"])
        if ratio >= DEGRADING_MULTIPLIER:
            return "DEGRADING"
        if ratio >= WARNING_MULTIPLIER:
            return "WARNING"
        return "HEALTHY"

    forecast_monitoring["Model_Status"] = forecast_monitoring.apply(forecast_status, axis=1)
    display(forecast_monitoring)
else:
    forecast_monitoring = pd.DataFrame()
    print("Forecast metrics unavailable.")


,Machine_ID,Sensor,Train_Rows,Test_Rows,MAE,RMSE,MAE_Ratio_To_Baseline,RMSE_Ratio_To_Baseline,Model_Status
0,CNC_001,Temperature_C,200,50,5.097954,5.970408,3.538924,3.586682,DEGRADING
1,CNC_001,Vibration_mm_s,200,50,0.347378,0.407473,0.241144,0.244787,HEALTHY
2,CNC_001,Pressure_bar,200,50,0.034376,0.041368,0.023863,0.024851,HEALTHY
3,COMP_003,Temperature_C,224,56,4.774992,5.529774,3.314729,3.321974,DEGRADING
4,COMP_003,Vibration_mm_s,224,56,0.389138,0.446527,0.270134,0.268248,HEALTHY
5,COMP_003,Pressure_bar,224,56,0.032507,0.039267,0.022566,0.023590,HEALTHY
6,MOTOR_004,Temperature_C,184,46,3.667262,4.119010,2.545759,2.474467,DEGRADING
7,MOTOR_004,Vibration_mm_s,184,46,0.105852,0.119998,0.073481,0.072088,HEALTHY
8,MOTOR_004,Pressure_bar,184,46,0.057638,0.068639,0.040011,0.041234,HEALTHY
9,PRESS_006,Temperature_C,212,53,2.170568,2.526785,1.506776,1.517949,DEGRADING


## STEP 3 — RUL monitoring

If the Smart Twin artifact contains both actual and predicted RUL, calculate MAE and RMSE.

If either value is absent, the notebook reports that RUL monitoring cannot be recalculated from the saved artifact.


In [4]:
def find_col(data, candidates):
    if data is None:
        return None
    for c in candidates:
        if c in data.columns:
            return c
    return None

rul_actual_col = find_col(
    smart_twin, ["RUL_Cycles", "actual_rul_cycles", "Actual_RUL", "Actual_RUL_Cycles"]
)
rul_pred_col = find_col(
    smart_twin, ["predicted_rul_cycles", "Predicted_RUL", "Predicted_RUL_Cycles", "RUL_Prediction"]
)

if smart_twin is not None and rul_actual_col and rul_pred_col:
    rul_data = smart_twin[[rul_actual_col, rul_pred_col]].dropna()
    rul_mae = mean_absolute_error(rul_data[rul_actual_col], rul_data[rul_pred_col])
    rul_rmse = np.sqrt(mean_squared_error(rul_data[rul_actual_col], rul_data[rul_pred_col]))
    print("RUL MAE:", round(rul_mae, 4))
    print("RUL RMSE:", round(rul_rmse, 4))
else:
    rul_mae = np.nan
    rul_rmse = np.nan
    print("RUL actual/predicted columns are not both available.")


RUL actual/predicted columns are not both available.


## STEP 4 — Failure prediction monitoring

When actual and predicted failure labels are available, calculate:
- Accuracy
- Precision
- Recall
- F1

F1 is particularly useful when failures are less frequent than normal observations.


In [5]:
failure_actual_col = find_col(
    smart_twin, ["Failure", "failure", "Actual_Failure", "actual_failure"]
)
failure_pred_col = find_col(
    smart_twin, ["predicted_failure", "Predicted_Failure", "failure_prediction", "Failure_Prediction"]
)

failure_metrics = {}

if smart_twin is not None and failure_actual_col and failure_pred_col:
    failure_data = smart_twin[[failure_actual_col, failure_pred_col]].dropna()
    y_true = failure_data[failure_actual_col].astype(int)
    y_pred = failure_data[failure_pred_col].astype(int)

    failure_metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }
    display(pd.DataFrame([failure_metrics]))
else:
    print("Actual and predicted failure columns are not both available.")


Actual and predicted failure columns are not both available.


## STEP 5 — Feature/data drift

Compare reference and current feature distributions using the Kolmogorov–Smirnov test.

A low p-value indicates evidence of distribution shift. Drift is an indicator, not proof that the model is inaccurate.


In [6]:
DRIFT_ALPHA = 0.05

candidate_features = [
    "Temperature_C", "Vibration_mm_s", "Pressure_bar",
    "Rotational_Speed_RPM", "Load_Fraction", "Torque_Nm",
    "Current_A", "Acoustic_dB", "Wear_Level",
    "Health_Index", "Anomaly_Score"
]

available_features = [
    f for f in candidate_features
    if reference_df is not None and smart_twin is not None
    and f in reference_df.columns and f in smart_twin.columns
]

drift_records = []

for feature in available_features:
    ref = pd.to_numeric(reference_df[feature], errors="coerce").dropna()
    cur = pd.to_numeric(smart_twin[feature], errors="coerce").dropna()

    if len(ref) < 2 or len(cur) < 2:
        continue

    statistic, p_value = ks_2samp(ref, cur)

    drift_records.append({
        "Feature": feature,
        "Reference_Mean": ref.mean(),
        "Current_Mean": cur.mean(),
        "KS_Statistic": statistic,
        "P_Value": p_value,
        "Drift_Detected": p_value < DRIFT_ALPHA
    })

feature_drift = pd.DataFrame(drift_records)
display(feature_drift)


""


## STEP 6 — Prediction drift

Where failure-probability predictions exist in both reference and current data, compare their distributions as well.


In [7]:
failure_prob_col = find_col(
    smart_twin, ["failure_probability", "Failure_Probability", "failure_prob"]
)
reference_prob_col = find_col(
    reference_df, ["failure_probability", "Failure_Probability", "failure_prob"]
)

prediction_drift = pd.DataFrame()

if failure_prob_col and reference_prob_col:
    ref = pd.to_numeric(reference_df[reference_prob_col], errors="coerce").dropna()
    cur = pd.to_numeric(smart_twin[failure_prob_col], errors="coerce").dropna()

    if len(ref) >= 2 and len(cur) >= 2:
        statistic, p_value = ks_2samp(ref, cur)
        prediction_drift = pd.DataFrame([{
            "Prediction": "Failure Probability",
            "Reference_Mean": ref.mean(),
            "Current_Mean": cur.mean(),
            "KS_Statistic": statistic,
            "P_Value": p_value,
            "Drift_Detected": p_value < DRIFT_ALPHA
        }])

display(prediction_drift)


""


## STEP 7 — Visualize representative drift

In [8]:
for feature in [f for f in ["Temperature_C", "Vibration_mm_s", "Pressure_bar"] if f in available_features]:
    plt.figure(figsize=(10, 4))
    plt.hist(reference_df[feature].dropna(), bins=30, alpha=0.5, label="Reference")
    plt.hist(smart_twin[feature].dropna(), bins=30, alpha=0.5, label="Current")
    plt.title(f"Reference vs Current — {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()


## STEP 8 — Determine overall model health

The status combines the available evidence.

This is a monitoring demonstration, so the thresholds are explicit and configurable. With real production data, they should be calibrated using historical validation and business risk.


In [9]:
FAILURE_F1_WARNING = 0.75
FAILURE_F1_DEGRADING = 0.60

drift_count = int(feature_drift["Drift_Detected"].sum()) if not feature_drift.empty else 0
prediction_drift_detected = (
    bool(prediction_drift["Drift_Detected"].iloc[0])
    if not prediction_drift.empty else False
)

forecast_degrading_count = (
    int((forecast_monitoring["Model_Status"] == "DEGRADING").sum())
    if not forecast_monitoring.empty else 0
)
forecast_warning_count = (
    int((forecast_monitoring["Model_Status"] == "WARNING").sum())
    if not forecast_monitoring.empty else 0
)

failure_f1 = failure_metrics.get("F1", np.nan)
degrading_reasons = []
warning_reasons = []

if not np.isnan(failure_f1):
    if failure_f1 < FAILURE_F1_DEGRADING:
        degrading_reasons.append("Failure-model F1 is below the degradation threshold.")
    elif failure_f1 < FAILURE_F1_WARNING:
        warning_reasons.append("Failure-model F1 is below the warning threshold.")

if forecast_degrading_count:
    degrading_reasons.append(
        f"{forecast_degrading_count} time-series model(s) exceed the degradation threshold."
    )
elif forecast_warning_count:
    warning_reasons.append(
        f"{forecast_warning_count} time-series model(s) exceed the warning threshold."
    )

if drift_count:
    warning_reasons.append(
        f"{drift_count} feature distribution(s) show statistically significant drift."
    )

if prediction_drift_detected:
    warning_reasons.append("Failure-probability prediction distribution shows drift.")

if degrading_reasons:
    overall_model_status = "DEGRADING"
elif warning_reasons:
    overall_model_status = "WARNING"
else:
    overall_model_status = "HEALTHY"

print("Overall Model Status:", overall_model_status)
print("\nDegrading reasons:")
for r in degrading_reasons:
    print("-", r)

print("\nWarning reasons:")
for r in warning_reasons:
    print("-", r)


Overall Model Status: DEGRADING

Degrading reasons:
- 6 time-series model(s) exceed the degradation threshold.

Warning reasons:


## STEP 9 — Create monitoring summary

In [10]:
summary = {
    "overall_model_status": overall_model_status,
    "failure_model_f1": failure_f1,
    "rul_mae": rul_mae,
    "rul_rmse": rul_rmse,
    "time_series_average_mae": (
        forecast_metrics["MAE"].mean()
        if forecast_metrics is not None and not forecast_metrics.empty else np.nan
    ),
    "time_series_average_rmse": (
        forecast_metrics["RMSE"].mean()
        if forecast_metrics is not None and not forecast_metrics.empty else np.nan
    ),
    "feature_drift_count": drift_count,
    "prediction_drift_detected": prediction_drift_detected,
    "retraining_recommended": overall_model_status in ["WARNING", "DEGRADING"]
}

summary_df = pd.DataFrame([summary])
display(summary_df)


,overall_model_status,failure_model_f1,rul_mae,rul_rmse,time_series_average_mae,time_series_average_rmse,feature_drift_count,prediction_drift_detected,retraining_recommended
0,DEGRADING,NaN,NaN,NaN,1.440538,1.664605,0,False,True


## STEP 10 — Save monitoring artifacts

These outputs will later feed the Executive Dashboard:

- `model_monitoring_metrics.csv`
- `feature_drift_report.csv`
- `prediction_drift_report.csv`
- `model_monitoring_summary.csv`


In [11]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

metric_rows = []

for metric, value in failure_metrics.items():
    metric_rows.append({"Model": "Failure Prediction", "Metric": metric, "Value": value})

if not np.isnan(rul_mae):
    metric_rows.extend([
        {"Model": "RUL Prediction", "Metric": "MAE", "Value": rul_mae},
        {"Model": "RUL Prediction", "Metric": "RMSE", "Value": rul_rmse}
    ])

if forecast_metrics is not None and not forecast_metrics.empty:
    metric_rows.extend([
        {"Model": "Time-Series Forecasting", "Metric": "Average MAE", "Value": forecast_metrics["MAE"].mean()},
        {"Model": "Time-Series Forecasting", "Metric": "Average RMSE", "Value": forecast_metrics["RMSE"].mean()}
    ])

monitoring_metrics_df = pd.DataFrame(metric_rows)

paths = {
    "metrics": f"{OUTPUT_DIR}/model_monitoring_metrics.csv",
    "feature_drift": f"{OUTPUT_DIR}/feature_drift_report.csv",
    "prediction_drift": f"{OUTPUT_DIR}/prediction_drift_report.csv",
    "summary": f"{OUTPUT_DIR}/model_monitoring_summary.csv"
}

monitoring_metrics_df.to_csv(paths["metrics"], index=False)
feature_drift.to_csv(paths["feature_drift"], index=False)
prediction_drift.to_csv(paths["prediction_drift"], index=False)
summary_df.to_csv(paths["summary"], index=False)

for key, path in paths.items():
    print(key, "->", path)


metrics -> ../data/processed/model_monitoring_metrics.csv
feature_drift -> ../data/processed/feature_drift_report.csv
prediction_drift -> ../data/processed/prediction_drift_report.csv
summary -> ../data/processed/model_monitoring_summary.csv


## STEP 11 — Human-readable monitoring report

In [12]:
print("=" * 65)
print("MODEL MONITORING REPORT")
print("=" * 65)
print("Overall Model Status :", overall_model_status)
print("Failure Model F1     :", "N/A" if np.isnan(failure_f1) else round(failure_f1, 4))
print("RUL MAE              :", "N/A" if np.isnan(rul_mae) else round(rul_mae, 4))
print("RUL RMSE             :", "N/A" if np.isnan(rul_rmse) else round(rul_rmse, 4))

if forecast_metrics is not None and not forecast_metrics.empty:
    print("Time-Series MAE      :", round(forecast_metrics["MAE"].mean(), 4))
    print("Time-Series RMSE     :", round(forecast_metrics["RMSE"].mean(), 4))

print("Feature Drift Count  :", drift_count)
print("Prediction Drift     :", "YES" if prediction_drift_detected else "NO")
print("Retraining Recommended:",
      "YES" if summary["retraining_recommended"] else "NO")

if overall_model_status == "DEGRADING":
    print("\nACTION: Investigate model degradation and schedule retraining.")
elif overall_model_status == "WARNING":
    print("\nACTION: Continue close monitoring and investigate drift/performance changes.")
else:
    print("\nACTION: Continue normal model monitoring.")


MODEL MONITORING REPORT
Overall Model Status : DEGRADING
Failure Model F1     : N/A
RUL MAE              : N/A
RUL RMSE             : N/A
Time-Series MAE      : 1.4405
Time-Series RMSE     : 1.6646
Feature Drift Count  : 0
Prediction Drift     : NO
Retraining Recommended: YES

ACTION: Investigate model degradation and schedule retraining.


## STEP 12 — Final validation

A WARNING or DEGRADING result is **not a notebook failure**. It means the monitoring system detected a condition that deserves investigation.


In [13]:
print("=" * 65)
print("MODEL MONITORING VALIDATION")
print("=" * 65)
print("Overall status:", overall_model_status)

for key, path in paths.items():
    print(f"{key} exists:", os.path.exists(path))
    assert os.path.exists(path)

assert overall_model_status in ["HEALTHY", "WARNING", "DEGRADING"]

print("\nSTATUS: MODEL MONITORING PIPELINE COMPLETED SUCCESSFULLY")


MODEL MONITORING VALIDATION
Overall status: DEGRADING
metrics exists: True
feature_drift exists: True
prediction_drift exists: True
summary exists: True

STATUS: MODEL MONITORING PIPELINE COMPLETED SUCCESSFULLY


# End of Notebook

## What this component adds

### Machine health

```text
Failure probability
RUL
Health Index
Anomaly Score
        ↓
Machine condition
```

### Model health

```text
Prediction performance
Forecasting error
Feature drift
Prediction drift
        ↓
Model condition
        ↓
Retraining recommendation
```

The next major phase is **deployment + the local Executive Dashboard**.

The dashboard will eventually show the simulated IoT stream, machine health, sensor forecasts, agent decisions, orchestrator events, Memory context, RAG evidence, LLM report, and model-monitoring status.
